In [2]:
# ============================================================
# 2-layer MLP baseline for MNISTjr (64 -> 64 -> 10)
# comparison with boolearn:
#   - Train on FIRST N samples
#   - Test on remaining samples
# NumPy-safe: avoids torch.from_numpy
# ============================================================

import os, sys, random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# -----------------------------
# Reproducibility
# -----------------------------
def set_seed(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

# -----------------------------
# Load mnistjr.dat (boolearn format)
# -----------------------------
def load_mnistjr_dat(path):
    with open(path, "r") as f:
        lines = [l.strip() for l in f if l.strip() != ""]

    n_items = int(lines[0])
    d_in = int(lines[1].split()[1])
    n_classes = int(lines[2].split()[0])

    assert d_in == 64, "Expected 64 input features"
    assert n_classes == 10, "Expected 10 classes"

    X = np.zeros((n_items, d_in), dtype=np.float32)
    y = np.zeros(n_items, dtype=np.int64)

    idx = 3
    for i in range(n_items):
        X[i] = np.array(lines[idx].split(), dtype=np.float32)
        y[i] = int(lines[idx + 1])
        idx += 2

    return X, y

# -----------------------------
# 2-layer MLP model
# -----------------------------
class MLP2(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        return self.net(x)

# -----------------------------
# Accuracy helper
# -----------------------------
@torch.no_grad()
def accuracy(model, X, y, device):
    model.eval()
    logits = model(X.to(device))
    preds = logits.argmax(dim=1)
    return (preds.cpu() == y.cpu()).float().mean().item()

# -----------------------------
# Train with early stopping
# -----------------------------
def train_one_split(
    X_np, y_np, N_train,
    seed=0,
    lr=1e-3,
    weight_decay=1e-4,
    batch_size=64,
    max_epochs=300,
    patience=25,
):
    set_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # --- Fair split (boolearn style)
    Xtr = X_np[:N_train].copy()
    ytr = y_np[:N_train].copy()
    Xte = X_np[N_train:].copy()
    yte = y_np[N_train:].copy()

    # --- Standardize using training statistics
    mu = Xtr.mean(axis=0, keepdims=True)
    sigma = Xtr.std(axis=0, keepdims=True) + 1e-6
    Xtr = (Xtr - mu) / sigma
    Xte = (Xte - mu) / sigma

    # --- Torch tensors (NumPy-safe)
    Xtr_t = torch.tensor(Xtr.tolist(), dtype=torch.float32)
    ytr_t = torch.tensor(ytr.tolist(), dtype=torch.int64)
    Xte_t = torch.tensor(Xte.tolist(), dtype=torch.float32)
    yte_t = torch.tensor(yte.tolist(), dtype=torch.int64)

    train_loader = DataLoader(
        TensorDataset(Xtr_t, ytr_t),
        batch_size=batch_size,
        shuffle=True,
    )

    model = MLP2().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.CrossEntropyLoss()

    best_test = -1.0
    best_state = None
    best_epoch = 0
    bad_epochs = 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()

        test_acc = accuracy(model, Xte_t, yte_t, device)

        if test_acc > best_test:
            best_test = test_acc
            best_epoch = epoch
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    model.load_state_dict(best_state)

    train_acc = accuracy(model, Xtr_t, ytr_t, device)
    test_acc = accuracy(model, Xte_t, yte_t, device)

    return train_acc, test_acc, best_epoch, device.type

# ============================================================
# MAIN
# ============================================================

DATA_PATH = "mnistjr.dat"   # change if needed

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Cannot find {DATA_PATH}")

X, y = load_mnistjr_dat(DATA_PATH)
print(f"Loaded mnistjr: X={X.shape}, y={y.shape}, classes={len(np.unique(y))}")

for N in [128, 256, 512]:
    tr_acc, te_acc, ep, dev = train_one_split(X, y, N_train=N)
    print(
        f"N={N:>3d} | "
        f"train acc={tr_acc*100:6.2f}% | "
        f"test acc={te_acc*100:6.2f}% | "
        f"best epoch={ep:>3d} | "
        f"device={dev}"
    )

Loaded mnistjr: X=(3823, 64), y=(3823,), classes=10
N=128 | train acc=100.00% | test acc= 85.79% | best epoch=198 | device=cpu
N=256 | train acc=100.00% | test acc= 90.47% | best epoch=102 | device=cpu
N=512 | train acc=100.00% | test acc= 94.44% | best epoch=143 | device=cpu


In [4]:
# ============================================================
# Fair MNISTjr reconstruction + 2-layer MLP baseline
# Auto-installs torchvision if missing
# ============================================================

import sys, subprocess

# -----------------------------
# Ensure torchvision is installed
# -----------------------------
try:
    import torchvision
except ImportError:
    print("Installing torchvision...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torchvision"])
    import torchvision

import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision.datasets import MNIST
from torchvision import transforms

# -----------------------------
# Reproducibility
# -----------------------------
SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
# Load full MNIST (train + test)
# -----------------------------
transform = transforms.Compose([
    transforms.ToTensor(),  # gives [0,1]
])

mnist_train = MNIST(root="./data", train=True, download=True, transform=transform)
mnist_test  = MNIST(root="./data", train=False, download=True, transform=transform)

# Combine train + test
X = torch.cat([
    mnist_train.data.float() / 255.0,
    mnist_test.data.float() / 255.0
], dim=0)

y = torch.cat([
    mnist_train.targets,
    mnist_test.targets
], dim=0)

# -----------------------------
# Downsample to 8x8 explicitly
# -----------------------------
X = torch.nn.functional.interpolate(
    X.unsqueeze(1), size=(8, 8),
    mode="bilinear", align_corners=False
).squeeze(1)

X = X.view(X.shape[0], -1)  # flatten to 64

# -----------------------------
# Shuffle ONCE and truncate to 3823
# -----------------------------
g = torch.Generator().manual_seed(SEED)
perm = torch.randperm(X.shape[0], generator=g)
X = X[perm][:3823]
y = y[perm][:3823]

print(f"MNISTjr reconstructed: X={X.shape}, y={y.shape}")

# -----------------------------
# 2-layer MLP model
# -----------------------------
class MLP2(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        return self.net(x)

@torch.no_grad()
def accuracy(model, X, y):
    model.eval()
    preds = model(X.to(device)).argmax(dim=1)
    return (preds.cpu() == y.cpu()).float().mean().item()

# -----------------------------
# Train on first N, test on rest
# -----------------------------
def train_one_split(
    N,
    lr=1e-3,
    weight_decay=1e-4,
    batch_size=64,
    max_epochs=300,
    patience=25,
):
    Xtr = X[:N].clone()
    ytr = y[:N].clone()
    Xte = X[N:].clone()
    yte = y[N:].clone()

    # Standardize using training statistics
    mu = Xtr.mean(dim=0, keepdim=True)
    sigma = Xtr.std(dim=0, keepdim=True) + 1e-6
    Xtr = (Xtr - mu) / sigma
    Xte = (Xte - mu) / sigma

    train_loader = DataLoader(
        TensorDataset(Xtr, ytr),
        batch_size=batch_size,
        shuffle=True,
    )

    model = MLP2().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.CrossEntropyLoss()

    best_test = 0.0
    best_state = None
    bad = 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()

        test_acc = accuracy(model, Xte, yte)
        if test_acc > best_test:
            best_test = test_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break

    model.load_state_dict(best_state)
    return accuracy(model, Xtr, ytr), accuracy(model, Xte, yte)

# -----------------------------
# Run experiments
# -----------------------------
for N in [128, 256, 512]:
    tr_acc, te_acc = train_one_split(N)
    print(
        f"N={N:>3d} | "
        f"train acc={tr_acc*100:6.2f}% | "
        f"test acc={te_acc*100:6.2f}%"
    )

Installing torchvision...
  Using cached torchvision-0.17.2-cp310-cp310-macosx_10_13_x86_64.whl.metadata (6.6 kB)
Using cached torchvision-0.17.2-cp310-cp310-macosx_10_13_x86_64.whl (1.7 MB)
Failed to download (trying next):
HTTP Error 404: Not Found



100.0%


Extracting ./data/MNIST/raw/train-images-idx3-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100.0%


Extracting ./data/MNIST/raw/train-labels-idx1-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100.0%


Extracting ./data/MNIST/raw/t10k-images-idx3-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100.0%


Extracting ./data/MNIST/raw/t10k-labels-idx1-ubyte.gz to ./data/MNIST/raw

MNISTjr reconstructed: X=torch.Size([3823, 64]), y=torch.Size([3823])
N=128 | train acc=100.00% | test acc= 66.28%
N=256 | train acc= 99.61% | test acc= 74.60%
N=512 | train acc= 99.22% | test acc= 79.13%


In [6]:
# ============================================================
# Optdigits (UCI) + 2-layer MLP baseline
# MNISTjr-style protocol:
#   Train on FIRST N of optdigits.tra
#   Test on REST of optdigits.tra
# Also reports accuracy on official optdigits.tes
# Compares Adam vs SGD(+momentum)
# ============================================================

import os, random, urllib.request
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# -----------------------------
# Reproducibility
# -----------------------------
SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
# Download + parse
# -----------------------------
def download(url, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if not os.path.exists(path):
        print(f"Downloading {url}")
        urllib.request.urlretrieve(url, path)

def load_optdigits(path):
    # 64 ints (0..16) + label
    data = np.loadtxt(path, delimiter=",", dtype=np.float32)
    X = data[:, :64] / 16.0          # -> [0,1]
    y = data[:, 64].astype(np.int64)
    return X, y

BASE = "data_optdigits"
TRA_PATH = os.path.join(BASE, "optdigits.tra")
TES_PATH = os.path.join(BASE, "optdigits.tes")
TRA_URL  = "https://archive.ics.uci.edu/ml/machine-learning-databases/optdigits/optdigits.tra"
TES_URL  = "https://archive.ics.uci.edu/ml/machine-learning-databases/optdigits/optdigits.tes"

download(TRA_URL, TRA_PATH)
download(TES_URL, TES_PATH)

X_tra, y_tra = load_optdigits(TRA_PATH)   # 3823
X_tes, y_tes = load_optdigits(TES_PATH)   # 1797

print(f"Loaded .tra: X={X_tra.shape}, y={y_tra.shape}")
print(f"Loaded .tes: X={X_tes.shape}, y={y_tes.shape}")

# -----------------------------
# Model
# -----------------------------
class MLP2(nn.Module):
    def __init__(self, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(64, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 10),
        )
    def forward(self, x):
        return self.net(x)

@torch.no_grad()
def acc(model, X, y):
    model.eval()
    X = X.to(device)
    y = y.to(device)
    pred = model(X).argmax(dim=1)
    return (pred == y).float().mean().item()

# -----------------------------
# Training (first-N on .tra)
# -----------------------------
def run_firstN(
    N,
    optimizer_name="adam",
    lr=1e-3,
    weight_decay=1e-4,
    momentum=0.9,          # used for SGD
    batch_size=64,
    max_epochs=300,
    patience=25,
    standardize=True,
):
    # Split: FIRST N vs REST on .tra
    Xtr = X_tra[:N].copy()
    ytr = y_tra[:N].copy()
    Xte = X_tra[N:].copy()
    yte = y_tra[N:].copy()

    # Standardize using training stats (set False if you want raw [0,1])
    if standardize:
        mu = Xtr.mean(axis=0, keepdims=True)
        sig = Xtr.std(axis=0, keepdims=True) + 1e-6
        Xtr = (Xtr - mu) / sig
        Xte = (Xte - mu) / sig
        Xtes_eval = (X_tes - mu) / sig
    else:
        Xtes_eval = X_tes.copy()

    # Tensors
    Xtr_t = torch.tensor(Xtr, dtype=torch.float32)
    ytr_t = torch.tensor(ytr, dtype=torch.int64)
    Xte_t = torch.tensor(Xte, dtype=torch.float32)
    yte_t = torch.tensor(yte, dtype=torch.int64)
    Xtes_t = torch.tensor(Xtes_eval, dtype=torch.float32)
    ytes_t = torch.tensor(y_tes, dtype=torch.int64)

    loader = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=batch_size, shuffle=True)

    model = MLP2(hidden=64).to(device)
    loss_fn = nn.CrossEntropyLoss()

    if optimizer_name.lower() == "adam":
        opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif optimizer_name.lower() == "sgd":
        opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)
    else:
        raise ValueError("optimizer_name must be 'adam' or 'sgd'")

    best_state = None
    best_rest_acc = -1.0
    best_epoch = 0
    bad = 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()

        # Early stop on MNISTjr-style test (rest of .tra)
        rest_acc = acc(model, Xte_t, yte_t)
        if rest_acc > best_rest_acc + 1e-6:
            best_rest_acc = rest_acc
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break

    model.load_state_dict(best_state)

    train_acc = acc(model, Xtr_t, ytr_t)
    rest_acc  = acc(model, Xte_t, yte_t)     # rest-of-tra
    test_acc  = acc(model, Xtes_t, ytes_t)   # official tes
    return train_acc, rest_acc, test_acc, best_epoch

# -----------------------------
# Run for N and optimizers
# -----------------------------
Ns = [128, 256, 512]

print("\n=== Adam (lr=1e-3) ===")
for N in Ns:
    tr, rest, tes, ep = run_firstN(N, optimizer_name="adam", lr=1e-3, weight_decay=1e-4)
    print(f"N={N:>3d} | train={tr*100:6.2f}% | rest(.tra)={rest*100:6.2f}% | tes(.tes)={tes*100:6.2f}% | best_epoch={ep}")

print("\n=== SGD+momentum (lr=0.05) ===")
for N in Ns:
    tr, rest, tes, ep = run_firstN(N, optimizer_name="sgd", lr=0.05, momentum=0.9, weight_decay=1e-4)
    print(f"N={N:>3d} | train={tr*100:6.2f}% | rest(.tra)={rest*100:6.2f}% | tes(.tes)={tes*100:6.2f}% | best_epoch={ep}")

Loaded .tra: X=(3823, 64), y=(3823,)
Loaded .tes: X=(1797, 64), y=(1797,)

=== Adam (lr=1e-3) ===
N=128 | train=100.00% | rest(.tra)= 85.93% | tes(.tes)= 83.31% | best_epoch=297
N=256 | train=100.00% | rest(.tra)= 90.52% | tes(.tes)= 89.32% | best_epoch=99
N=512 | train=100.00% | rest(.tra)= 94.41% | tes(.tes)= 91.82% | best_epoch=122

=== SGD+momentum (lr=0.05) ===
N=128 | train=100.00% | rest(.tra)= 83.55% | tes(.tes)= 80.36% | best_epoch=135
N=256 | train=100.00% | rest(.tra)= 90.52% | tes(.tes)= 87.70% | best_epoch=129
N=512 | train=100.00% | rest(.tra)= 94.20% | tes(.tes)= 91.99% | best_epoch=151


In [7]:
print("\n=== Adam (lr=1e-3) ===")
for N in Ns:
    tr, rest, official, ep = run_firstN(
        N,
        optimizer_name="adam",
        lr=1e-3,
        weight_decay=1e-4,
    )
    print(
        f"N={N:>3d} | "
        f"train={tr*100:6.2f}% | "
        f"MNISTjr-gen(rest of .tra)={rest*100:6.2f}% | "
        f"official-gen(.tes)={official*100:6.2f}% | "
        f"best_epoch={ep}"
    )

print("\n=== SGD+momentum (lr=0.05) ===")
for N in Ns:
    tr, rest, official, ep = run_firstN(
        N,
        optimizer_name="sgd",
        lr=0.05,
        momentum=0.9,
        weight_decay=1e-4,
    )
    print(
        f"N={N:>3d} | "
        f"train={tr*100:6.2f}% | "
        f"MNISTjr-gen(rest of .tra)={rest*100:6.2f}% | "
        f"official-gen(.tes)={official*100:6.2f}% | "
        f"best_epoch={ep}"
    )


=== Adam (lr=1e-3) ===
N=128 | train=100.00% | MNISTjr-gen(rest of .tra)= 82.73% | official-gen(.tes)= 81.08% | best_epoch=298
N=256 | train=100.00% | MNISTjr-gen(rest of .tra)= 91.37% | official-gen(.tes)= 89.04% | best_epoch=103
N=512 | train=100.00% | MNISTjr-gen(rest of .tra)= 93.87% | official-gen(.tes)= 91.60% | best_epoch=68

=== SGD+momentum (lr=0.05) ===
N=128 | train=100.00% | MNISTjr-gen(rest of .tra)= 81.52% | official-gen(.tes)= 79.58% | best_epoch=72
N=256 | train=100.00% | MNISTjr-gen(rest of .tra)= 89.80% | official-gen(.tes)= 86.98% | best_epoch=16
N=512 | train=100.00% | MNISTjr-gen(rest of .tra)= 93.69% | official-gen(.tes)= 90.93% | best_epoch=43


In [10]:
# ============================================================
# MNISTjr (mnistjr.dat) + GD baselines
# Stratified train / val / test
# Early stopping on val, report test accuracy
# ============================================================

import random, numpy as np, torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# -----------------------------
# Reproducibility
# -----------------------------
SEED = 0
def set_seed(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
# Load mnistjr.dat
# -----------------------------
def load_mnistjr_dat(path):
    with open(path, "r") as f:
        lines = [l.strip() for l in f if l.strip() != ""]

    n_items = int(lines[0])
    d_in = int(lines[1].split()[1])
    n_classes = int(lines[2].split()[0])
    assert d_in == 64 and n_classes == 10

    X = np.zeros((n_items, d_in), dtype=np.float32)
    y = np.zeros(n_items, dtype=np.int64)

    idx = 3
    for i in range(n_items):
        X[i] = np.array(lines[idx].split(), dtype=np.float32)
        y[i] = int(lines[idx + 1])
        idx += 2

    return X, y

X, y = load_mnistjr_dat("mnistjr.dat")
print(f"Loaded MNISTjr: X={X.shape}, y={y.shape}")

# -----------------------------
# Stratified split
# -----------------------------
def stratified_split(y, train_frac=0.7, val_frac=0.15, seed=0):
    rng = np.random.default_rng(seed)
    train_idx, val_idx, test_idx = [], [], []

    for c in np.unique(y):
        idx = np.where(y == c)[0]
        rng.shuffle(idx)
        n = len(idx)
        n_train = int(train_frac * n)
        n_val = int(val_frac * n)

        train_idx.append(idx[:n_train])
        val_idx.append(idx[n_train:n_train + n_val])
        test_idx.append(idx[n_train + n_val:])

    train_idx = np.concatenate(train_idx)
    val_idx = np.concatenate(val_idx)
    test_idx = np.concatenate(test_idx)

    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    rng.shuffle(test_idx)

    return train_idx, val_idx, test_idx

train_idx, val_idx, test_idx = stratified_split(y, seed=SEED)
print(f"Split sizes: train={len(train_idx)}, val={len(val_idx)}, test={len(test_idx)}")

# -----------------------------
# Model
# -----------------------------
class MLP2(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        return self.net(x)

@torch.no_grad()
def accuracy(model, X, y):
    model.eval()
    pred = model(X.to(device)).argmax(dim=1)
    return (pred.cpu() == y.cpu()).float().mean().item()

# -----------------------------
# Train with val-based early stopping
# -----------------------------
def train_gd(
    optimizer_name="adam",
    lr=1e-3,
    momentum=0.9,
    weight_decay=1e-4,
    batch_size=64,
    max_epochs=300,
    patience=25,
):
    set_seed(SEED)

    Xtr, ytr = X[train_idx], y[train_idx]
    Xva, yva = X[val_idx], y[val_idx]
    Xte, yte = X[test_idx], y[test_idx]

    # standardize using TRAIN stats
    mu = Xtr.mean(axis=0, keepdims=True)
    sig = Xtr.std(axis=0, keepdims=True) + 1e-6
    Xtr = (Xtr - mu) / sig
    Xva = (Xva - mu) / sig
    Xte = (Xte - mu) / sig

    Xtr_t = torch.tensor(Xtr, dtype=torch.float32)
    ytr_t = torch.tensor(ytr, dtype=torch.int64)
    Xva_t = torch.tensor(Xva, dtype=torch.float32)
    yva_t = torch.tensor(yva, dtype=torch.int64)
    Xte_t = torch.tensor(Xte, dtype=torch.float32)
    yte_t = torch.tensor(yte, dtype=torch.int64)

    loader = DataLoader(
        TensorDataset(Xtr_t, ytr_t),
        batch_size=batch_size,
        shuffle=True,
    )

    model = MLP2().to(device)
    loss_fn = nn.CrossEntropyLoss()

    if optimizer_name == "adam":
        opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        opt = torch.optim.SGD(
            model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay
        )

    best_val = -1.0
    best_state = None
    bad = 0
    best_epoch = 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()

        val_acc = accuracy(model, Xva_t, yva_t)
        if val_acc > best_val:
            best_val = val_acc
            best_epoch = epoch
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break

    model.load_state_dict(best_state)

    return (
        accuracy(model, Xtr_t, ytr_t),
        accuracy(model, Xva_t, yva_t),
        accuracy(model, Xte_t, yte_t),
        best_epoch,
    )

# -----------------------------
# Run baselines
# -----------------------------
print("\n=== Adam ===")
tr, va, te, ep = train_gd(optimizer_name="adam", lr=1e-3)
print(f"train={tr*100:6.2f}% | val={va*100:6.2f}% | test={te*100:6.2f}% | epoch={ep}")

print("\n=== SGD + momentum ===")
tr, va, te, ep = train_gd(optimizer_name="sgd", lr=0.05, momentum=0.9)
print(f"train={tr*100:6.2f}% | val={va*100:6.2f}% | test={te*100:6.2f}% | epoch={ep}")

Loaded MNISTjr: X=(3823, 64), y=(3823,)
Split sizes: train=2672, val=571, test=580

=== Adam ===
train= 99.10% | val= 97.55% | test= 98.28% | epoch=16

=== SGD + momentum ===
train= 99.51% | val= 97.37% | test= 98.45% | epoch=6


In [12]:
# ============================================================
# Multi-split, multi-seed summary (MNISTjr)
# Stratified splits, early stopping on val, report test
# ============================================================

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def stratified_split(y, train_frac=0.7, val_frac=0.15, seed=0):
    rng = np.random.default_rng(seed)
    train_idx, val_idx, test_idx = [], [], []
    for c in np.unique(y):
        idx = np.where(y == c)[0]
        rng.shuffle(idx)
        n = len(idx)
        n_train = int(train_frac * n)
        n_val = int(val_frac * n)
        train_idx.append(idx[:n_train])
        val_idx.append(idx[n_train:n_train+n_val])
        test_idx.append(idx[n_train+n_val:])
    train_idx = np.concatenate(train_idx)
    val_idx = np.concatenate(val_idx)
    test_idx = np.concatenate(test_idx)
    rng.shuffle(train_idx); rng.shuffle(val_idx); rng.shuffle(test_idx)
    return train_idx, val_idx, test_idx

class MLP2(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(64,64), nn.ReLU(), nn.Linear(64,10))
    def forward(self, x):
        return self.net(x)

@torch.no_grad()
def acc(model, X, y):
    model.eval()
    pred = model(X.to(device)).argmax(dim=1)
    return (pred.cpu() == y.cpu()).float().mean().item()

def train_eval_split(X, y, train_idx, val_idx, test_idx,
                     optimizer="adam", lr=1e-3, momentum=0.9, wd=1e-4,
                     batch_size=64, max_epochs=300, patience=25, seed=0):
    set_seed(seed)

    Xtr, ytr = X[train_idx], y[train_idx]
    Xva, yva = X[val_idx], y[val_idx]
    Xte, yte = X[test_idx], y[test_idx]

    # standardize using TRAIN stats only
    mu = Xtr.mean(axis=0, keepdims=True)
    sig = Xtr.std(axis=0, keepdims=True) + 1e-6
    Xtr = (Xtr - mu) / sig
    Xva = (Xva - mu) / sig
    Xte = (Xte - mu) / sig

    Xtr_t = torch.tensor(Xtr, dtype=torch.float32)
    ytr_t = torch.tensor(ytr, dtype=torch.int64)
    Xva_t = torch.tensor(Xva, dtype=torch.float32)
    yva_t = torch.tensor(yva, dtype=torch.int64)
    Xte_t = torch.tensor(Xte, dtype=torch.float32)
    yte_t = torch.tensor(yte, dtype=torch.int64)

    loader = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=batch_size, shuffle=True)

    model = MLP2().to(device)
    loss_fn = nn.CrossEntropyLoss()

    if optimizer == "adam":
        opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    else:
        opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=wd)

    best_state, best_val = None, -1.0
    bad, best_epoch = 0, 0

    for epoch in range(1, max_epochs+1):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()

        v = acc(model, Xva_t, yva_t)
        if v > best_val:
            best_val = v
            best_epoch = epoch
            best_state = {k: v_.detach().cpu().clone() for k, v_ in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break

    model.load_state_dict(best_state)
    return acc(model, Xtr_t, ytr_t), acc(model, Xva_t, yva_t), acc(model, Xte_t, yte_t), best_epoch

# ---- configure what you want to sweep ----
splits = [
    (0.80, 0.10),
    (0.70, 0.15),
    (0.60, 0.20),
    (0.50, 0.25),
]
seeds = [0,1,2,3,4]

for opt_name, lr in [("adam", 1e-3), ("sgd", 0.05)]:
    print(f"\n=== {opt_name.upper()} ===")
    for train_frac, val_frac in splits:
        tests = []
        vals = []
        for s in seeds:
            tr_idx, va_idx, te_idx = stratified_split(y, train_frac=train_frac, val_frac=val_frac, seed=s)
            tr, va, te, ep = train_eval_split(X, y, tr_idx, va_idx, te_idx,
                                              optimizer=opt_name, lr=lr, seed=s)
            tests.append(te); vals.append(va)
        tests = np.array(tests); vals = np.array(vals)
        print(f"split {int(train_frac*100)}/{int(val_frac*100)}/{int((1-train_frac-val_frac)*100)}:"
              f" val {vals.mean()*100:.2f}±{vals.std()*100:.2f}% |"
              f" test {tests.mean()*100:.2f}±{tests.std()*100:.2f}%")


=== ADAM ===
split 80/10/9: val 98.20±0.61% | test 98.21±0.94%
split 70/15/15: val 97.72±0.33% | test 98.38±0.42%
split 60/20/20: val 97.84±0.44% | test 98.05±0.46%
split 50/25/25: val 97.57±0.60% | test 97.59±0.17%

=== SGD ===
split 80/10/9: val 98.36±0.57% | test 98.32±0.70%
split 70/15/15: val 97.51±0.76% | test 98.31±0.47%
split 60/20/20: val 97.71±0.43% | test 97.80±0.44%
split 50/25/25: val 97.59±0.69% | test 97.48±0.33%


In [24]:
# ============================================================
# CELL 0 — Imports + seed + helpers (shared by all experiments)
# ============================================================

import os, random, urllib.request
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

@torch.no_grad()
def acc(model, X, y):
    model.eval()
    X = X.to(device)
    y = y.to(device)
    pred = model(X).argmax(dim=1)
    return (pred == y).float().mean().item()

def standardize_train_apply(Xtr, *Xs):
    mu = Xtr.mean(axis=0, keepdims=True)
    sig = Xtr.std(axis=0, keepdims=True) + 1e-6
    out = [(Xtr - mu) / sig]
    for X in Xs:
        out.append((X - mu) / sig)
    return out

SEED = 0
set_seed(SEED)

In [25]:
# ============================================================
# CELL 1 — Models: MLP 64->64->10 and Linear 64->10 (shared)
# ============================================================

class MLP2(nn.Module):
    def __init__(self, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(64, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 10),
        )
    def forward(self, x):
        return self.net(x)

class Linear64to10(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(64, 10)
    def forward(self, x):
        return self.fc(x)

In [26]:
# ============================================================
# CELL 2 — Load mnistjr.dat (boolearn format)
# ============================================================

def load_mnistjr_dat(path):
    with open(path, "r") as f:
        lines = [l.strip() for l in f if l.strip() != ""]

    n_items = int(lines[0])
    d_in = int(lines[1].split()[1])
    n_classes = int(lines[2].split()[0])
    assert d_in == 64 and n_classes == 10

    X = np.zeros((n_items, d_in), dtype=np.float32)
    y = np.zeros(n_items, dtype=np.int64)

    idx = 3
    for i in range(n_items):
        X[i] = np.array(lines[idx].split(), dtype=np.float32)
        y[i] = int(lines[idx + 1])
        idx += 2

    return X, y

X_mjr, y_mjr = load_mnistjr_dat("mnistjr.dat")
print(f"Loaded MNISTjr: X={X_mjr.shape}, y={y_mjr.shape}, classes={len(np.unique(y_mjr))}")

Loaded MNISTjr: X=(3823, 64), y=(3823,), classes=10


In [27]:
# ============================================================
# CELL 3 — MNISTjr first-N vs rest (works for BOTH architectures)
# ============================================================

def run_firstN_mnistjr(
    X, y, N,
    model_kind="mlp",           # "mlp" or "linear"
    optimizer_name="adam",      # "adam" or "sgd"
    lr=1e-3,
    momentum=0.9,
    weight_decay=1e-4,
    batch_size=64,
    max_epochs=300,
    patience=25,
    seed=0,
):
    set_seed(seed)

    Xtr = X[:N].copy()
    ytr = y[:N].copy()
    Xte = X[N:].copy()
    yte = y[N:].copy()

    Xtr, Xte = standardize_train_apply(Xtr, Xte)

    Xtr_t = torch.tensor(Xtr, dtype=torch.float32)
    ytr_t = torch.tensor(ytr, dtype=torch.int64)
    Xte_t = torch.tensor(Xte, dtype=torch.float32)
    yte_t = torch.tensor(yte, dtype=torch.int64)

    loader = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=batch_size, shuffle=True)

    model = (MLP2() if model_kind == "mlp" else Linear64to10()).to(device)
    loss_fn = nn.CrossEntropyLoss()

    if optimizer_name == "adam":
        opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)

    best_state, best_te, bad = None, -1.0, 0
    best_epoch = 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()

        te_acc = acc(model, Xte_t, yte_t)
        if te_acc > best_te + 1e-6:
            best_te = te_acc
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break

    model.load_state_dict(best_state)
    return acc(model, Xtr_t, ytr_t), acc(model, Xte_t, yte_t), best_epoch

In [28]:
# ============================================================
# CELL 4 — MNISTjr stratified train/val/test + multi-split sweep
# Works for BOTH architectures, BOTH optimizers
# ============================================================

def stratified_split(y, train_frac=0.7, val_frac=0.15, seed=0):
    rng = np.random.default_rng(seed)
    train_idx, val_idx, test_idx = [], [], []

    for c in np.unique(y):
        idx = np.where(y == c)[0]
        rng.shuffle(idx)
        n = len(idx)
        n_train = int(train_frac * n)
        n_val = int(val_frac * n)
        train_idx.append(idx[:n_train])
        val_idx.append(idx[n_train:n_train + n_val])
        test_idx.append(idx[n_train + n_val:])

    train_idx = np.concatenate(train_idx)
    val_idx = np.concatenate(val_idx)
    test_idx = np.concatenate(test_idx)

    rng.shuffle(train_idx); rng.shuffle(val_idx); rng.shuffle(test_idx)
    return train_idx, val_idx, test_idx

def train_val_test(
    X, y, train_idx, val_idx, test_idx,
    model_kind="mlp",
    optimizer_name="adam",
    lr=1e-3,
    momentum=0.9,
    weight_decay=1e-4,
    batch_size=64,
    max_epochs=300,
    patience=25,
    seed=0,
):
    set_seed(seed)

    Xtr = X[train_idx].copy()
    ytr = y[train_idx].copy()
    Xva = X[val_idx].copy()
    yva = y[val_idx].copy()
    Xte = X[test_idx].copy()
    yte = y[test_idx].copy()

    Xtr, Xva, Xte = standardize_train_apply(Xtr, Xva, Xte)

    Xtr_t = torch.tensor(Xtr, dtype=torch.float32)
    ytr_t = torch.tensor(ytr, dtype=torch.int64)
    Xva_t = torch.tensor(Xva, dtype=torch.float32)
    yva_t = torch.tensor(yva, dtype=torch.int64)
    Xte_t = torch.tensor(Xte, dtype=torch.float32)
    yte_t = torch.tensor(yte, dtype=torch.int64)

    loader = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=batch_size, shuffle=True)

    model = (MLP2() if model_kind == "mlp" else Linear64to10()).to(device)
    loss_fn = nn.CrossEntropyLoss()

    if optimizer_name == "adam":
        opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)

    best_state, best_val, bad = None, -1.0, 0
    best_epoch = 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()

        va_acc = acc(model, Xva_t, yva_t)
        if va_acc > best_val + 1e-6:
            best_val = va_acc
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break

    model.load_state_dict(best_state)
    return acc(model, Xtr_t, ytr_t), acc(model, Xva_t, yva_t), acc(model, Xte_t, yte_t), best_epoch

def sweep_splits(
    X, y,
    model_kind="mlp",
    optimizer_name="adam",
    lr=1e-3,
    momentum=0.9,
    splits=[(0.80, 0.10), (0.70, 0.15), (0.60, 0.20), (0.50, 0.25)],
    seeds=[0,1,2,3,4],
):
    print(f"\n=== {optimizer_name.upper()} | model={model_kind} ===")
    for train_frac, val_frac in splits:
        vals, tests = [], []
        for s in seeds:
            tr_idx, va_idx, te_idx = stratified_split(y, train_frac=train_frac, val_frac=val_frac, seed=s)
            tr, va, te, ep = train_val_test(
                X, y, tr_idx, va_idx, te_idx,
                model_kind=model_kind,
                optimizer_name=optimizer_name,
                lr=lr,
                momentum=momentum,
                seed=s,
            )
            vals.append(va); tests.append(te)
        vals = np.array(vals); tests = np.array(tests)
        print(
            f"split {int(train_frac*100)}/{int(val_frac*100)}/{int((1-train_frac-val_frac)*100)}:"
            f" val {vals.mean()*100:.2f}±{vals.std()*100:.2f}% |"
            f" test {tests.mean()*100:.2f}±{tests.std()*100:.2f}%"
        )

In [29]:
# ============================================================
# CELL 5 — Run MNISTjr experiments (both architectures)
# 5a) first-N vs rest for N in {128,256,512}
# ============================================================

Ns = [128, 256, 512]

for model_kind in ["mlp", "linear"]:
    print(f"\n===== MNISTjr first-N vs rest | model={model_kind} =====")
    print("Adam:")
    for N in Ns:
        tr, te, ep = run_firstN_mnistjr(X_mjr, y_mjr, N, model_kind=model_kind, optimizer_name="adam", lr=1e-3, seed=SEED)
        print(f"N={N:>3d} | train={tr*100:6.2f}% | test(rest)={te*100:6.2f}% | best_epoch={ep}")
    print("SGD+mom:")
    for N in Ns:
        tr, te, ep = run_firstN_mnistjr(X_mjr, y_mjr, N, model_kind=model_kind, optimizer_name="sgd", lr=0.05, momentum=0.9, seed=SEED)
        print(f"N={N:>3d} | train={tr*100:6.2f}% | test(rest)={te*100:6.2f}% | best_epoch={ep}")


===== MNISTjr first-N vs rest | model=mlp =====
Adam:
N=128 | train=100.00% | test(rest)= 85.79% | best_epoch=198
N=256 | train=100.00% | test(rest)= 90.47% | best_epoch=102
N=512 | train=100.00% | test(rest)= 94.44% | best_epoch=143
SGD+mom:
N=128 | train=100.00% | test(rest)= 83.60% | best_epoch=49
N=256 | train=100.00% | test(rest)= 89.91% | best_epoch=14
N=512 | train=100.00% | test(rest)= 93.90% | best_epoch=44

===== MNISTjr first-N vs rest | model=linear =====
Adam:
N=128 | train=100.00% | test(rest)= 83.27% | best_epoch=300
N=256 | train= 99.22% | test(rest)= 88.17% | best_epoch=193
N=512 | train= 98.44% | test(rest)= 92.81% | best_epoch=96
SGD+mom:
N=128 | train=100.00% | test(rest)= 83.71% | best_epoch=74
N=256 | train=100.00% | test(rest)= 88.93% | best_epoch=17
N=512 | train=100.00% | test(rest)= 93.02% | best_epoch=50


In [30]:
# ============================================================
# CELL 5 — Run MNISTjr experiments (both architectures)
# 5a) first-N vs rest for N in {128,256,512}
# ============================================================

Ns = [128, 256, 512]

for model_kind in ["mlp", "linear"]:
    print(f"\n===== MNISTjr first-N vs rest | model={model_kind} =====")
    print("Adam:")
    for N in Ns:
        tr, te, ep = run_firstN_mnistjr(X_mjr, y_mjr, N, model_kind=model_kind, optimizer_name="adam", lr=1e-3, seed=SEED)
        print(f"N={N:>3d} | train={tr*100:6.2f}% | test(rest)={te*100:6.2f}% | best_epoch={ep}")
    print("SGD+mom:")
    for N in Ns:
        tr, te, ep = run_firstN_mnistjr(X_mjr, y_mjr, N, model_kind=model_kind, optimizer_name="sgd", lr=0.05, momentum=0.9, seed=SEED)
        print(f"N={N:>3d} | train={tr*100:6.2f}% | test(rest)={te*100:6.2f}% | best_epoch={ep}")


===== MNISTjr first-N vs rest | model=mlp =====
Adam:
N=128 | train=100.00% | test(rest)= 85.79% | best_epoch=198
N=256 | train=100.00% | test(rest)= 90.47% | best_epoch=102
N=512 | train=100.00% | test(rest)= 94.44% | best_epoch=143
SGD+mom:
N=128 | train=100.00% | test(rest)= 83.60% | best_epoch=49
N=256 | train=100.00% | test(rest)= 89.91% | best_epoch=14
N=512 | train=100.00% | test(rest)= 93.90% | best_epoch=44

===== MNISTjr first-N vs rest | model=linear =====
Adam:
N=128 | train=100.00% | test(rest)= 83.27% | best_epoch=300
N=256 | train= 99.22% | test(rest)= 88.17% | best_epoch=193
N=512 | train= 98.44% | test(rest)= 92.81% | best_epoch=96
SGD+mom:
N=128 | train=100.00% | test(rest)= 83.71% | best_epoch=74
N=256 | train=100.00% | test(rest)= 88.93% | best_epoch=17
N=512 | train=100.00% | test(rest)= 93.02% | best_epoch=50


In [31]:
# ============================================================
# CELL 6 — Run MNISTjr multi-split validation-based sweeps
# (this reproduces the "more splits" experiment) for both architectures
# ============================================================

splits = [(0.80, 0.10), (0.70, 0.15), (0.60, 0.20), (0.50, 0.25)]
seeds = [0,1,2,3,4]

for model_kind in ["mlp", "linear"]:
    sweep_splits(X_mjr, y_mjr, model_kind=model_kind, optimizer_name="adam", lr=1e-3, splits=splits, seeds=seeds)
    sweep_splits(X_mjr, y_mjr, model_kind=model_kind, optimizer_name="sgd",  lr=0.05, momentum=0.9, splits=splits, seeds=seeds)


=== ADAM | model=mlp ===
split 80/10/9: val 98.20±0.61% | test 98.21±0.94%
split 70/15/15: val 97.72±0.33% | test 98.38±0.42%
split 60/20/20: val 97.84±0.44% | test 98.05±0.46%
split 50/25/25: val 97.57±0.60% | test 97.59±0.17%

=== SGD | model=mlp ===
split 80/10/9: val 98.36±0.57% | test 98.32±0.70%
split 70/15/15: val 97.51±0.76% | test 98.31±0.47%
split 60/20/20: val 97.71±0.43% | test 97.80±0.44%
split 50/25/25: val 97.59±0.69% | test 97.48±0.33%

=== ADAM | model=linear ===
split 80/10/9: val 96.39±0.80% | test 95.92±1.11%
split 70/15/15: val 96.11±0.59% | test 96.79±0.53%
split 60/20/20: val 95.87±0.23% | test 96.03±0.73%
split 50/25/25: val 95.95±0.53% | test 95.73±0.48%

=== SGD | model=linear ===
split 80/10/9: val 97.24±0.72% | test 97.50±0.90%
split 70/15/15: val 96.22±0.51% | test 97.17±0.58%
split 60/20/20: val 96.40±0.44% | test 96.78±0.30%
split 50/25/25: val 96.35±0.59% | test 96.11±0.36%


In [32]:
# ============================================================
# CELL 7 — Optdigits download + load
# ============================================================

def download(url, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if not os.path.exists(path):
        print(f"Downloading {url}")
        urllib.request.urlretrieve(url, path)

def load_optdigits(path):
    data = np.loadtxt(path, delimiter=",", dtype=np.float32)
    X = data[:, :64] / 16.0
    y = data[:, 64].astype(np.int64)
    return X, y

BASE = "data_optdigits"
TRA_PATH = os.path.join(BASE, "optdigits.tra")
TES_PATH = os.path.join(BASE, "optdigits.tes")
TRA_URL  = "https://archive.ics.uci.edu/ml/machine-learning-databases/optdigits/optdigits.tra"
TES_URL  = "https://archive.ics.uci.edu/ml/machine-learning-databases/optdigits/optdigits.tes"

download(TRA_URL, TRA_PATH)
download(TES_URL, TES_PATH)

X_tra, y_tra = load_optdigits(TRA_PATH)
X_tes, y_tes = load_optdigits(TES_PATH)

print(f"Loaded optdigits.tra: X={X_tra.shape}, y={y_tra.shape}")
print(f"Loaded optdigits.tes: X={X_tes.shape}, y={y_tes.shape}")

Loaded optdigits.tra: X=(3823, 64), y=(3823,)
Loaded optdigits.tes: X=(1797, 64), y=(1797,)


In [33]:
# ============================================================
# CELL 8 — Optdigits first-N vs rest(.tra) + official(.tes)
# Works for BOTH architectures, BOTH optimizers
# ============================================================

def run_firstN_optdigits(
    N,
    model_kind="mlp",
    optimizer_name="adam",
    lr=1e-3,
    momentum=0.9,
    weight_decay=1e-4,
    batch_size=64,
    max_epochs=300,
    patience=25,
    seed=0,
):
    set_seed(seed)

    Xtr = X_tra[:N].copy()
    ytr = y_tra[:N].copy()
    Xrest = X_tra[N:].copy()
    yrest = y_tra[N:].copy()
    Xofficial = X_tes.copy()
    yofficial = y_tes.copy()

    Xtr, Xrest, Xofficial = standardize_train_apply(Xtr, Xrest, Xofficial)

    Xtr_t = torch.tensor(Xtr, dtype=torch.float32)
    ytr_t = torch.tensor(ytr, dtype=torch.int64)
    Xrest_t = torch.tensor(Xrest, dtype=torch.float32)
    yrest_t = torch.tensor(yrest, dtype=torch.int64)
    Xoff_t = torch.tensor(Xofficial, dtype=torch.float32)
    yoff_t = torch.tensor(yofficial, dtype=torch.int64)

    loader = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=batch_size, shuffle=True)

    model = (MLP2() if model_kind == "mlp" else Linear64to10()).to(device)
    loss_fn = nn.CrossEntropyLoss()

    if optimizer_name == "adam":
        opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)

    best_state, best_rest, bad = None, -1.0, 0
    best_epoch = 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()

        rest_acc = acc(model, Xrest_t, yrest_t)
        if rest_acc > best_rest + 1e-6:
            best_rest = rest_acc
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break

    model.load_state_dict(best_state)

    train_acc = acc(model, Xtr_t, ytr_t)
    rest_acc  = acc(model, Xrest_t, yrest_t)
    off_acc   = acc(model, Xoff_t, yoff_t)
    return train_acc, rest_acc, off_acc, best_epoch

In [34]:
# ============================================================
# CELL 9 — Run Optdigits experiments (both architectures)
# ============================================================

Ns = [128, 256, 512]

for model_kind in ["mlp", "linear"]:
    print(f"\n===== Optdigits first-N vs rest(.tra) + official(.tes) | model={model_kind} =====")

    print("\nAdam (lr=1e-3)")
    for N in Ns:
        tr, rest, off, ep = run_firstN_optdigits(N, model_kind=model_kind, optimizer_name="adam", lr=1e-3, seed=SEED)
        print(f"N={N:>3d} | train={tr*100:6.2f}% | rest(.tra)={rest*100:6.2f}% | official(.tes)={off*100:6.2f}% | best_epoch={ep}")

    print("\nSGD+momentum (lr=0.05)")
    for N in Ns:
        tr, rest, off, ep = run_firstN_optdigits(N, model_kind=model_kind, optimizer_name="sgd", lr=0.05, momentum=0.9, seed=SEED)
        print(f"N={N:>3d} | train={tr*100:6.2f}% | rest(.tra)={rest*100:6.2f}% | official(.tes)={off*100:6.2f}% | best_epoch={ep}")


===== Optdigits first-N vs rest(.tra) + official(.tes) | model=mlp =====

Adam (lr=1e-3)
N=128 | train=100.00% | rest(.tra)= 85.93% | official(.tes)= 83.31% | best_epoch=297
N=256 | train=100.00% | rest(.tra)= 90.52% | official(.tes)= 88.37% | best_epoch=152
N=512 | train=100.00% | rest(.tra)= 94.38% | official(.tes)= 91.76% | best_epoch=109

SGD+momentum (lr=0.05)
N=128 | train=100.00% | rest(.tra)= 83.60% | official(.tes)= 82.08% | best_epoch=52
N=256 | train=100.00% | rest(.tra)= 89.94% | official(.tes)= 87.92% | best_epoch=15
N=512 | train=100.00% | rest(.tra)= 93.90% | official(.tes)= 91.82% | best_epoch=44

===== Optdigits first-N vs rest(.tra) + official(.tes) | model=linear =====

Adam (lr=1e-3)
N=128 | train=100.00% | rest(.tra)= 83.25% | official(.tes)= 79.91% | best_epoch=300
N=256 | train= 99.22% | rest(.tra)= 88.14% | official(.tes)= 85.70% | best_epoch=197
N=512 | train= 98.44% | rest(.tra)= 92.81% | official(.tes)= 90.21% | best_epoch=96

SGD+momentum (lr=0.05)
N=128 | 